In [ ]:
#########################################################################
# # ! Moon Topographic-Gravitational-Potential Computation: PH vs SH
#########################################################################
# %% 
# # ! Setup
import pandas as pd
import pyshtools as pysh
import pyvista as pv
import xarray as xr
import time
from datetime import datetime
from gravity_forward_numba import *

In [ ]:
# # ! Start time
print("=" * 80)
print(f"Start time: [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]")
print("=" * 80)

Start time: [2026-05-18 20:00:16]


In [ ]:
# # ! Functions
def Shgrid2Mesh(shgrid):
    mesh = pv.Sphere(radius=1.0, center=(0.0, 0.0, 0.0),
                     theta_resolution=shgrid.nlon-1,
                     phi_resolution=shgrid.nlat)
    [LON, LAT] = np.meshgrid(shgrid.lons(), shgrid.lats())
    R = shgrid.data
    X = R * np.cos(np.deg2rad(LAT)) * np.cos(np.deg2rad(LON))
    Y = R * np.cos(np.deg2rad(LAT)) * np.sin(np.deg2rad(LON))
    Z = R * np.sin(np.deg2rad(LAT))
    # north pole, south pole, and regular points
    pts_x = np.hstack([X[0,0], X[-1,0], X[1:-1,:-1].flatten(order='F')]) 
    pts_y = np.hstack([Y[0,0], Y[-1,0], Y[1:-1,:-1].flatten(order='F')])
    pts_z = np.hstack([Z[0,0], Z[-1,0], Z[1:-1,:-1].flatten(order='F')])
    pts_r = np.sqrt(pts_x**2 + pts_y**2 + pts_z**2)
    Verts = np.column_stack((pts_x, pts_y, pts_z))
    mesh.points = Verts
    return mesh, pts_r

In [ ]:
# # ! Load Top-N largest |gD| errors
df_errors = pd.read_csv('output/Moon_gD_errors.csv')
err_lat = df_errors['Lat_deg'].values
err_lon = df_errors['Lon_deg'].values
err_lat_rad = np.deg2rad(err_lat)
err_lon_rad = np.deg2rad(err_lon)
gD_SH = df_errors['gD_SH'].values
df_SH_PHs = df_errors[['Lat_deg', 'Lon_deg', 'gD_SH']].copy()

In [ ]:
# # ! Computation of TGP: Polyhedron
#### * Shape of Moon
lmax_shp = 359
clm_shp_moon = \
    pysh.SHCoeffs.from_file(f'input/Moon_shape_719.sh', 
                            lmax=lmax_shp, 
                            name='LOLA_shape (Moon)',
                            units='m', format='bshc')
r_itfc_km = clm_shp_moon.coeffs[0,0,0] / 1.e3
#### * Calculation points
rho0 = 2560.0 # kg/m^3
r_calc_km = 1748000.0 / 1.e3
xps = r_calc_km * np.cos(err_lat_rad) * np.cos(err_lon_rad)
yps = r_calc_km * np.cos(err_lat_rad) * np.sin(err_lon_rad)
zps = r_calc_km * np.sin(err_lat_rad)
#### * Computation of TGP
IN_RES = [15.0/60.0, 6.0/60.0, 3.0/60.0, 2.0/60.0, 1.0/60.0] # degree
for in_res in IN_RES:
    nc_Topo = f'output/moon_topo_Lshp{lmax_shp}_{int(60*in_res)}arcmin.nc'
    grd_topo_moon = pysh.SHGrid.from_netcdf(nc_Topo)
    grd_shp_moon = grd_topo_moon + r_itfc_km
    mesh_shp_moon, _ = Shgrid2Mesh(grd_shp_moon)
    mesh_shp_moon
    Verts = mesh_shp_moon.points
    Faces = mesh_shp_moon.regular_faces
    del mesh_shp_moon
    P = np.column_stack((xps, yps, zps))
    t0 = time.time()
    vgt_PH = WerSch_numba_v2(P, Verts, Faces, rho0)
    tc = time.time() - t0
    print("Computation info: \n"
         f"Number of computation points: {P.shape[0]} \n"
         "Polyhedron geometry: \n"
         f"Input resolution: {int(60*in_res):2d}-arcmin \n"
         f"Number of faces: {Faces.shape[0]} \n"
         f"Number of vertices: {Verts.shape[0]} \n"
         f"Time cost : {tc:8.3f} sec = {tc/60:.3f} min = {tc/(60*60):.3f} hr\n")
    #### * TGP = Shape model gravity - Sphere (r=r_itfc) gravity
    vgt_SP = gsphere(xps, yps, zps, 0, 0, 0, r_itfc_km, rho0)
    vgt_TP = tuple(g_PH - g_SP for g_PH, g_SP in zip(vgt_PH, vgt_SP))
    V, gx, gy, gz, Txx, Txy, Txz, Tyy, Tyz, Tzz = vgt_TP
    gN, gE, gD, TNN, TNE, TND, TEE, TED, TDD = \
        rotate_vec_ten_ecef2ned(err_lon_rad, err_lat_rad, 
                                gx, gy, gz, 
                                Txx, Txy, Txz, Tyy, Tyz, Tzz)
    df_SH_PHs[f'gD_PH_{int(60*in_res)}'] = np.round(gD, 4)
    df_SH_PHs[f'e_gD_PH_{int(60*in_res)}'] = np.round(gD - gD_SH, 4)

Computation info: 
Number of computation points: 10 
Polyhedron geometry: 
Input resolution: 15-arcmin 
Number of faces: 2070720 
Number of vertices: 1035362 
Time cost :    0.171 sec = 0.003 min = 0.000 hr

Computation info: 
Number of computation points: 10 
Polyhedron geometry: 
Input resolution:  6-arcmin 
Number of faces: 12952800 
Number of vertices: 6476402 
Time cost :    0.958 sec = 0.016 min = 0.000 hr

Computation info: 
Number of computation points: 10 
Polyhedron geometry: 
Input resolution:  3-arcmin 
Number of faces: 51825600 
Number of vertices: 25912802 
Time cost :    3.673 sec = 0.061 min = 0.001 hr

Computation info: 
Number of computation points: 10 
Polyhedron geometry: 
Input resolution:  2-arcmin 
Number of faces: 116618400 
Number of vertices: 58309202 
Time cost :    8.780 sec = 0.146 min = 0.002 hr

Computation info: 
Number of computation points: 10 
Polyhedron geometry: 
Input resolution:  1-arcmin 
Number of faces: 466516800 
Number of vertices: 233258402 

In [ ]:
# # ! End time
print("=" * 80)
print(f"End time: [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]")
print("=" * 80)

End time: [2026-05-18 20:02:11]


In [ ]:
print(df_SH_PHs)

   Lat_deg  Lon_deg     gD_SH  gD_PH_15  e_gD_PH_15   gD_PH_6  e_gD_PH_6  \
0   -22.50   204.00  694.1629  672.6280    -21.5349  690.8041    -3.3588   
1    -7.50   142.00  670.9713  654.8978    -16.0735  668.2368    -2.7345   
2     6.75   195.50  973.4527  958.6547    -14.7980  970.6598    -2.7929   
3   -15.00   177.75  610.4359  595.8525    -14.5834  607.9462    -2.4897   
4   -28.25   155.25  612.4432  599.3499    -13.0933  610.2635    -2.1797   
5    15.50   227.75  835.4962  822.7385    -12.7577  833.3320    -2.1642   
6   -10.75   225.50  879.6713  867.2452    -12.4261  877.5639    -2.1074   
7   -49.00   128.25  316.5163  304.4645    -12.0518  314.5028    -2.0135   
8    28.25   197.75  645.6960  634.1788    -11.5172  643.7104    -1.9856   
9   -35.75   220.50  337.4188  326.3042    -11.1146  335.5733    -1.8455   

    gD_PH_3  e_gD_PH_3   gD_PH_2  e_gD_PH_2   gD_PH_1  e_gD_PH_1  
0  693.9312    -0.2317  694.5203     0.3574  694.8746     0.7117  
1  670.4193    -0.5520  670.8